In [1]:
# Centralized imports (StarDist only)
from pathlib import Path
import time
import numpy as np
import pandas as pd
import tifffile
from tifffile import imwrite
from skimage.transform import rescale
import matplotlib.pyplot as plt

from csbdeep.utils import normalize
from stardist.models import StarDist2D, StarDist3D

# StarDist runs on TensorFlow. Native Windows TensorFlow GPU support ended at TF 2.10;
# for GPU acceleration on recent cards use WSL2 / Linux with `tensorflow[and-cuda]`.
import tensorflow as tf

gpus = tf.config.list_physical_devices("GPU")
status = {
    "tf_version": tf.__version__,
    "gpu_available": len(gpus) > 0,
    "gpus": [g.name for g in gpus],
}
print(status)


{'tf_version': '2.10.1', 'gpu_available': True, 'gpus': ['/physical_device:GPU:0']}


In [2]:

_MODEL_CACHE = {}  # cache loaded models across instances so we don't have to reload them for each image


def _iou_matrix(masks_a, masks_b):
    """Label-to-label intersection-over-union matrix between two 2D label images.

    Rows index labels of `masks_a` (0 = background), columns index labels of `masks_b`.
    """
    a = masks_a.ravel().astype(np.int64)
    b = masks_b.ravel().astype(np.int64)
    na = int(a.max()) + 1
    nb = int(b.max()) + 1
    overlap = np.zeros((na, nb), dtype=np.int64)
    np.add.at(overlap, (a, b), 1)
    n_a = overlap.sum(axis=1, keepdims=True)
    n_b = overlap.sum(axis=0, keepdims=True)
    union = n_a + n_b - overlap
    return np.where(union > 0, overlap / union, 0.0)


def _stitch_3d(masks, stitch_threshold=0.25):
    """Stitch a stack of 2D label planes (Z, Y, X) into a 3D labelling by IoU overlap
    between adjacent planes. Standalone reimplementation of the standard stitching used
    by cellpose, so this notebook has no cellpose dependency."""
    masks = masks.astype(np.int32).copy()
    if masks.shape[0] == 0:
        return masks
    mmax = int(masks[0].max())
    for i in range(masks.shape[0] - 1):
        iou = _iou_matrix(masks[i + 1], masks[i])[1:, 1:]  # drop background row/col
        icount = int(masks[i + 1].max())
        if iou.size == 0:
            if icount > 0:
                istitch = np.append(0, np.arange(mmax + 1, mmax + icount + 1))
                masks[i + 1] = istitch[masks[i + 1]]
                mmax += icount
            continue
        iou[iou < stitch_threshold] = 0.0
        iou[iou < iou.max(axis=0)] = 0.0
        istitch = iou.argmax(axis=1) + 1
        ino = np.nonzero(iou.max(axis=1) == 0.0)[0]
        istitch[ino] = np.arange(mmax + 1, mmax + len(ino) + 1)
        mmax += len(ino)
        istitch = np.append(0, istitch)
        masks[i + 1] = istitch[masks[i + 1]]
    return masks


class SegmentationComparisons:
    """Run StarDist nuclear segmentation (and parameter/model sweeps) on a 2-channel image.

    The input image is a 2-channel z-stack stored as (Z, C, Y, X), where
    channel `dapi_channel` is DAPI and channel `bf_channel` is brightfield/phase.
    Only the DAPI channel is used -- both segmentation and the overlay PNG ignore the
    brightfield channel. Every (method, parameter-combo) writes a label mask + overlay
    PNG to `output_dir` and contributes one row to the results table. The original/prior
    segmentations live on a network drive that is currently unavailable, so they are
    NOT loaded here -- the saved TIFs/PNGs can be compared against them later.

    OUTPUT LAYOUT (built to compare combos across all images):
        output_dir / <method> / <param-combo> / <image_name>.tif
        output_dir / <method> / <param-combo> / <image_name>.png
    So each combo folder collects the SAME model+parameter combo across every image,
    making it easy to scan one folder and judge which combo is consistently best.
    """

    def __init__(self, two_channel_image_path, scale_factor_xy=3, scale_factor_z=2,
        output_dir=Path(r"E:\stardist_comparisons"),
        dapi_channel=0, bf_channel=1):
        self.scale_factor_xy = float(scale_factor_xy)
        self.scale_factor_z = float(scale_factor_z)

        self.two_channel_image_path = Path(two_channel_image_path)
        self.dapi_channel = int(dapi_channel)
        self.bf_channel = int(bf_channel)
        self.image_name = self.two_channel_image_path.stem

        self.output_dir = Path(output_dir)

        # (Z, C, Y, X)
        self.two_channel_image = tifffile.imread(self.two_channel_image_path)
        if self.two_channel_image.ndim != 4:
            raise ValueError(f"Expected a 4D (Z, C, Y, X) image, got shape {self.two_channel_image.shape}")

        self.results = {}  # label -> label array
        self.counts = {}   # label -> object count

    def pixel_size(self):
        with tifffile.TiffFile(self.two_channel_image_path) as tif:
            tags = {tag.name: tag.value for tag in tif.pages[0].tags.values()}
            x_um = 1 / (tags["XResolution"][0] / tags["XResolution"][1])
            y_um = 1 / (tags["YResolution"][0] / tags["YResolution"][1])
            try:
                z_um = float(str(tags["IJMetadata"]).split("nscales=")[1].split(",")[2].split("\\nunit")[0])
            except Exception:
                z_um = float(str(tags["ImageDescription"]).split("spacing=")[1].split("loop")[0])
        self.original_spacing = (x_um, y_um, z_um)

    def rescale_image(self):
        """Downsample the DAPI channel and compute z-anisotropy (brightfield is ignored)."""
        self.pixel_size()
        x_um, y_um, z_um = self.original_spacing
        xy_ratio = 1.0 / self.scale_factor_xy
        z_ratio = 1.0 / self.scale_factor_z

        dapi = self.two_channel_image[:, self.dapi_channel].astype(np.float32)  # (Z, Y, X)

        self.dapi_ds = rescale(dapi, (z_ratio, xy_ratio, xy_ratio), anti_aliasing=True, preserve_range=True).astype(np.float32)

        # physical voxel spacing after downsampling
        self.z_spacing_ds = z_um * self.scale_factor_z
        self.xy_spacing_ds = x_um * self.scale_factor_xy
        self.anisotropy = self.z_spacing_ds / self.xy_spacing_ds
        self.pixel_size_um_2d = float(self.xy_spacing_ds)
        print(f"downsampled DAPI shape {self.dapi_ds.shape}, anisotropy {self.anisotropy:.3f}")

    # ---- model loaders (cached) ----

    @staticmethod
    def _get_stardist2d(model_name):
        key = f"sd2d_{model_name}"
        if key not in _MODEL_CACHE:
            _MODEL_CACHE[key] = StarDist2D.from_pretrained(model_name)
        return _MODEL_CACHE[key]

    @staticmethod
    def _get_stardist3d(model_name):
        key = f"sd3d_{model_name}"
        if key not in _MODEL_CACHE:
            _MODEL_CACHE[key] = StarDist3D.from_pretrained(model_name)
        return _MODEL_CACHE[key]

    # ---- segmentation methods: each takes its swept params and RETURNS a label volume ----

    def stardist_2d_stitched(self, model_name="2D_versatile_fluo", prob_thresh=None,
                             nms_thresh=None, stitch_threshold=0.1):
        """StarDist 2D applied per-z (DAPI only) then stitched into 3D by IoU overlap.
        `prob_thresh`/`nms_thresh` of None use the model's optimized defaults."""
        model = self._get_stardist2d(model_name)
        z_masks = []
        for z in range(self.dapi_ds.shape[0]):
            plane = normalize(self.dapi_ds[z], 1, 99.8)
            lab, _ = model.predict_instances(plane, prob_thresh=prob_thresh, nms_thresh=nms_thresh)
            z_masks.append(lab.astype(np.int32))
        stacked = np.stack(z_masks, axis=0)
        return _stitch_3d(stacked, stitch_threshold=stitch_threshold)

    def stardist_3d(self, model_name="3D_demo", prob_thresh=None, nms_thresh=None):
        """StarDist native-3D on the DAPI volume. NOTE: the only bundled 3D model is the
        '3D_demo' weights, so results are a baseline rather than production quality.

        The volume is tiled (`n_tiles`) so a single 3D conv doesn't have to fit the whole
        stack in GPU memory at once -- this avoids ResourceExhaustedError (OOM) on the GPU.
        StarDist's own heuristic picks an initial tiling, and we subdivide further (and
        retry) whenever the GPU still runs out of memory."""
        model = self._get_stardist3d(model_name)
        vol = normalize(self.dapi_ds, 1, 99.8)

        # let StarDist estimate a memory-safe tiling for this volume/model
        try:
            n_tiles = list(model._guess_n_tiles(vol))
        except Exception:
            n_tiles = [1, 1, 1]

        for _attempt in range(5):
            try:
                labels, _ = model.predict_instances(
                    vol, prob_thresh=prob_thresh, nms_thresh=nms_thresh, n_tiles=tuple(n_tiles))
                return labels
            except tf.errors.ResourceExhaustedError:
                # GPU OOM: subdivide further along Y and X (axes 1 and 2) and retry
                n_tiles[1] *= 2
                n_tiles[2] *= 2
                print(f"    3D OOM, retrying with more tiles {tuple(n_tiles)}")
        # let the error propagate if it still fails after the retries above
        labels, _ = model.predict_instances(
            vol, prob_thresh=prob_thresh, nms_thresh=nms_thresh, n_tiles=tuple(n_tiles))
        return labels

    # ---- output helpers ----

    @staticmethod
    def _filter_min_size(labels, min_size):
        """Remove objects smaller than `min_size` voxels (avoids counting debris)."""
        if not min_size:
            return labels
        ids, counts = np.unique(labels, return_counts=True)
        remove = ids[(ids != 0) & (counts < min_size)]
        if remove.size:
            labels = labels.copy()
            labels[np.isin(labels, remove)] = 0
        return labels

    @staticmethod
    def _param_label(params):
        """Short filesystem-safe string describing a parameter combo."""
        parts = [f"{k}={v}" for k, v in params.items()]
        label = "__".join(parts) if parts else "default"
        return label.replace(".", "p").replace("-", "neg")

    def _save_overlay_png(self, seg, png_path, method_name, suptitle):
        """Save a max-projection figure with two panels:
        DAPI, and the NEW StarDist labels over DAPI (brightfield is not used).
        """
        dapi_mip = self.dapi_ds.max(axis=0)
        new_label_mip = seg.max(axis=0) if seg.ndim == 3 else seg

        fig, axes = plt.subplots(1, 2, figsize=(10, 5))
        axes[0].imshow(dapi_mip, cmap="gray")
        axes[0].set_title("DAPI (max projection)")

        axes[1].imshow(dapi_mip, cmap="gray")
        new_overlay = np.ma.masked_where(new_label_mip == 0, new_label_mip)
        axes[1].imshow(new_overlay, cmap="nipy_spectral", alpha=0.5, interpolation="nearest")
        axes[1].set_title(f"NEW method: {method_name}")

        for ax in axes:
            ax.axis("off")
        fig.suptitle(suptitle)
        fig.tight_layout()
        fig.savefig(png_path, dpi=150, bbox_inches="tight")
        plt.close(fig)

    def _build_jobs(self, stardist2d_models, stardist3d_models,
                    prob_threshs_2d, nms_threshs_2d, stitch_thresholds,
                    prob_threshs_3d, nms_threshs_3d, include):
        """Expand the model/parameter grids into a flat list of (method, params, fn) jobs.
        2D and 3D have independent threshold grids so 2D can be kept small while 3D is
        explored more thoroughly."""
        jobs = []
        if "stardist_2dstitch" in include:
            for model_name in stardist2d_models:
                for stitch in stitch_thresholds:
                    for pt in prob_threshs_2d:
                        for nt in nms_threshs_2d:
                            jobs.append(("stardist_2dstitch",
                                         {"model_name": model_name, "stitch_threshold": stitch,
                                          "prob_thresh": pt, "nms_thresh": nt},
                                         self.stardist_2d_stitched))
        if "stardist_3d" in include:
            for model_name in stardist3d_models:
                for pt in prob_threshs_3d:
                    for nt in nms_threshs_3d:
                        jobs.append(("stardist_3d",
                                     {"model_name": model_name, "prob_thresh": pt, "nms_thresh": nt},
                                     self.stardist_3d))
        return jobs

    ######### MAIN PART############
    def run_sweep(self,
                  stardist2d_models=("2D_versatile_fluo",),
                  stardist3d_models=("3D_demo",),
                  prob_threshs_2d=(None,),
                  nms_threshs_2d=(None,),
                  stitch_thresholds=(0.1, 0.25),
                  prob_threshs_3d=(None, 0.4, 0.5),
                  nms_threshs_3d=(None, 0.3),
                  min_size=500,
                  include=("stardist_2dstitch", "stardist_3d")):
        """Run every (model, parameter-combo). Saves a TIF + overlay PNG per combo and
        returns one results row per successful run.

        2D is deliberately kept small (one model, default thresholds, a couple of stitch
        values); 3D gets a fuller prob/nms grid since it is expected to perform better.

        Outputs are grouped as output_dir/<method>/<param-combo>/<image_name>.{tif,png}
        so every combo folder gathers that exact combo across all images for easy
        side-by-side comparison."""
        self.rescale_image()
        jobs = self._build_jobs(stardist2d_models, stardist3d_models,
                                prob_threshs_2d, nms_threshs_2d, stitch_thresholds,
                                prob_threshs_3d, nms_threshs_3d, include)
        print(f"{self.image_name}: {len(jobs)} jobs queued")

        self.run_results = []
        for method_name, params, fn in jobs:
            param_label = self._param_label(params)
            label = f"{method_name}__{param_label}"
            try:
                t0 = time.time()
                seg = fn(**params)
                seg = self._filter_min_size(seg, min_size)
                time_taken = time.time() - t0

                count = int(np.unique(seg).size - (1 if (seg == 0).any() else 0))
                self.counts[label] = count
                self.results[label] = seg
                print(f"  {label}: {count} objects, {time_taken:.1f}s")

                # group by method/param-combo so the same combo across all images lands together
                combo_dir = self.output_dir / method_name / param_label
                combo_dir.mkdir(parents=True, exist_ok=True)
                imwrite(combo_dir / f"{self.image_name}.tif", seg.astype(np.uint32))
                self._save_overlay_png(seg, combo_dir / f"{self.image_name}.png",
                                       method_name, f"{self.image_name}__{label}")

                row = {"image_name": self.image_name, "method": method_name, "combo": label,
                       "time_taken": time_taken, "objects_found": count, "min_size": min_size}
                row.update(params)
                self.run_results.append(row)
            except Exception as exc:
                print(f"[SKIP] {label}: {type(exc).__name__}: {exc}")

        return self.run_results


In [3]:
# gather every 2-channel image copied to the local E drive
input_dir = Path(r"E:\two_channel_images")
image_paths = sorted(p for p in input_dir.iterdir() if p.suffix.lower() in (".tif", ".tiff"))
print(f"found {len(image_paths)} images in {input_dir}")
for p in image_paths:
    print(" ", p.name)


found 40 images in E:\two_channel_images
  dev10_1_day2_dapi_bf.tif
  dev10_3_day2_dapi_bf.tif
  dev10_4_day2_dapi_bf.tif
  dev10_4_day4_dapi_bf.tif
  dev10_day4_dapi_bf.tif
  dev1_1_P_1_dapi_bf.tif
  dev1_1_R_2_dapi_bf.tif
  dev1_1_R_3_dapi_bf.tif
  dev1_2_P1_dapi_bf.tif
  dev1_2_R_2_dapi_bf.tif
  dev1_2_R_3_dapi_bf.tif
  dev1_2_R_4_dapi_bf.tif
  dev1_2_R_5_dapi_bf.tif
  dev1_2_R_6_dapi_bf.tif
  dev4_1_6h_dapi_bf.tif
  dev4_1_R_2_dapi_bf.tif
  dev4_1_R_3_dapi_bf.tif
  dev4_1_R_4_dapi_bf.tif
  dev4_1_R_5_dapi_bf.tif
  dev4_2_6h_dapi_bf.tif
  dev4_3_6h_dapi_bf.tif
  dev4_3_P_1_dapi_bf.tif
  dev4_3_P_2_dapi_bf.tif
  dev4_3_R_2_dapi_bf.tif
  dev4_3_R_3_dapi_bf.tif
  dev4_3_R_4_dapi_bf.tif
  dev4_4_P_1_dapi_bf.tif
  dev4_4_P_2_dapi_bf.tif
  dev4_4_P_3_dapi_bf.tif
  dev4_4_R_2_dapi_bf.tif
  dev4_4_R_3_dapi_bf.tif
  dev4_4_R_4_dapi_bf.tif
  dev4_4_R_5_dapi_bf.tif
  dev4_4_R_6_dapi_bf.tif
  dev7_2_day1_dapi_bf.tif
  dev7_3_day1_dapi_bf.tif
  dev7_3_day1_large_dapi_bf.tif
  dev8_2_day1_noamp_d

In [ ]:
# run the StarDist sweep on every 2-channel image, collecting one results row per successful run
# save a per-image CSV as soon as each image finishes so an early termination doesn't lose data
# tuned sweep, kept under 20 combos/image: 2D sweeps prob/stitch, 3D sweeps prob
# higher prob_thresh -> only more-confident detections kept -> fewer (less spurious) objects.
# defaults are ~0.48 (2D_versatile_fluo) and ~0.71 (3D_demo), so we sweep clearly above those.
output_dir = Path(r"E:\stardist_comparisons")
sweep_kwargs = dict(
    stardist2d_models=("2D_versatile_fluo",),
    stardist3d_models=("3D_demo",),
    prob_threshs_2d=(0.5, 0.7, 0.85),
    nms_threshs_2d=(None,),
    stitch_thresholds=(0.1, 0.25),
    prob_threshs_3d=(0.5, 0.6,  0.65, 0.7, 0.),
    nms_threshs_3d=(None,),
    min_size=500,
    include=("stardist_2dstitch", "stardist_3d"),       
)
# 2D: 1 model x 3 prob x 1 nms x 2 stitch = 6 combos
# 3D: 1 model x 3 prob x 1 nms            = 3 combos
# total = 9 combos per image (< 20)

all_results = []
for image_path in image_paths:
    comparison = SegmentationComparisons(two_channel_image_path=image_path, output_dir=output_dir,
                                         scale_factor_xy=3, scale_factor_z=2)
    image_results = comparison.run_sweep(**sweep_kwargs)


downsampled DAPI shape (63, 1313, 1305), anisotropy 2.818
dev10_1_day2_dapi_bf: 11 jobs queued
Found model '2D_versatile_fluo' for 'StarDist2D'.
Loading network weights from 'weights_best.h5'.
Loading thresholds from 'thresholds.json'.
Using default values: prob_thresh=0.479071, nms_thresh=0.3.
  stardist_2dstitch__model_name=2D_versatile_fluo__stitch_threshold=0p1__prob_thresh=0p5__nms_thresh=None: 761 objects, 22.1s
  stardist_2dstitch__model_name=2D_versatile_fluo__stitch_threshold=0p1__prob_thresh=0p7__nms_thresh=None: 683 objects, 17.5s
  stardist_2dstitch__model_name=2D_versatile_fluo__stitch_threshold=0p1__prob_thresh=0p85__nms_thresh=None: 348 objects, 16.1s
  stardist_2dstitch__model_name=2D_versatile_fluo__stitch_threshold=0p25__prob_thresh=0p5__nms_thresh=None: 778 objects, 19.8s
  stardist_2dstitch__model_name=2D_versatile_fluo__stitch_threshold=0p25__prob_thresh=0p7__nms_thresh=None: 689 objects, 17.4s
  stardist_2dstitch__model_name=2D_versatile_fluo__stitch_threshold=0p2

100%|██████████| 242/242 [01:34<00:00,  2.56it/s]


  stardist_3d__model_name=3D_demo__prob_thresh=0p5__nms_thresh=None: 1131 objects, 258.7s


100%|██████████| 242/242 [01:35<00:00,  2.55it/s]


  stardist_3d__model_name=3D_demo__prob_thresh=0p6__nms_thresh=None: 918 objects, 153.6s


100%|██████████| 242/242 [01:35<00:00,  2.54it/s]


  stardist_3d__model_name=3D_demo__prob_thresh=0p65__nms_thresh=None: 736 objects, 131.0s


100%|██████████| 242/242 [01:35<00:00,  2.53it/s]


  stardist_3d__model_name=3D_demo__prob_thresh=0p7__nms_thresh=None: 501 objects, 112.2s


100%|██████████| 242/242 [02:02<00:00,  1.97it/s]


In [ ]:
# quick check that the tiled 3D path no longer OOMs (single image, 3D only, defaults)
_test = SegmentationComparisons(two_channel_image_path=image_paths[0],
                                output_dir=Path(r"E:\stardist_comparisons"))
_test_results = _test.run_sweep(include=("stardist_3d",),
                                prob_threshs_3d=(None,), nms_threshs_3d=(None,))
_test_results


downsampled DAPI shape (63, 1313, 1305), anisotropy 2.818
dev10_1_day2_dapi_bf: 1 jobs queued
Found model '3D_demo' for 'StarDist3D'.
Loading network weights from 'weights_best.h5'.
Loading thresholds from 'thresholds.json'.
Using default values: prob_thresh=0.707933, nms_thresh=0.3.


100%|██████████| 242/242 [01:34<00:00,  2.55it/s]


  stardist_3d__model_name=3D_demo__prob_thresh=None__nms_thresh=None: 462 objects, 113.5s


[{'image_name': 'dev10_1_day2_dapi_bf',
  'method': 'stardist_3d',
  'combo': 'stardist_3d__model_name=3D_demo__prob_thresh=None__nms_thresh=None',
  'time_taken': 113.47459101676941,
  'objects_found': 462,
  'min_size': 500,
  'model_name': '3D_demo',
  'prob_thresh': None,
  'nms_thresh': None}]